Import the required modules and functions

In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
import os
import re
import csv
from datetime import datetime

In [ ]:
# To create an empty csv file if it does not exists already, uncomment the below code

#header = ['ImageID','Title','Date']

#with open('BingImages.csv','w',newline='', encoding='UTF8') as f:
#    writer = csv.writer(f)
#    writer.writerow(header)

Download the homepage and find the number of pages in the website

In [ ]:
homepage_url = 'https://windows10spotlight.com/'

homepage_url_data = requests.get(homepage_url).text

homepage_soup = BeautifulSoup(homepage_url_data,"html.parser")

number_of_pages = int(re.findall('\d+.\d+', homepage_soup.find_all("nav", {"class":"navigation pagination"})[0].text)[1])

home_directory = os.getcwd()

bing_df = pd.read_csv('BingImages.csv')
bing_df

Download each page in the website to collect the image id's in each of it

In [ ]:
def clean_caption(caption):
    return caption.replace('/',' ').replace('\\',' ')

def download_image_pc(link, caption, extension):

    file_name = caption + '.' + extension
    
    link_data = requests.get(link)

    os.chdir(r'.\Bing Images\PC Version')
    path=os.path.join(os.getcwd() , file_name)
    
    with open(path, 'wb') as f:
        f.write(link_data.content)
    
    os.chdir(home_directory)

def download_image_mobile(link, caption, extension):

    file_name = caption + '.' + extension
    
    link_data = requests.get(link)

    os.chdir(r'.\Bing Images\Mobile Version')
    path=os.path.join(os.getcwd() , file_name)
    
    with open(path, 'wb') as f:
        f.write(link_data.content)
        print(file_name)
        
    os.chdir(home_directory)
    
def redirect_to_image(image_id):

    image_page_url = 'https://windows10spotlight.com/images/' + str(image_id)

    image_page_url_data = requests.get(image_page_url).text
    
    image_page_soup = BeautifulSoup(image_page_url_data, 'html.parser')

    #extract the date of the image
    dateStr = image_page_soup.find_all("span", {"class":"date"})[0].text.replace('-',' ')
    date = datetime.strptime(dateStr, '%d %b %Y').date()


    for link in image_page_soup.find_all('figure'):

        #clean the title
        image_caption = clean_caption(link.figcaption.text)

        #pc version
        pc_image_link = link.a.get('href')
        pc_image_extension = pc_image_link.split('.')[-1]
        download_image_pc(pc_image_link, image_caption, pc_image_extension)

        #mobile version
        parent_link = link.parent
        mobile_image_link = parent_link.findAll('p')[1].a.get('href')
        mobile_image_extension = mobile_image_link.split('.')[-1]
        download_image_mobile(mobile_image_link, image_caption, mobile_image_extension)
    
    #write the image_stamp to the csv
    data = [image_id, image_caption, date]
    with open('BingImages.csv','a+',newline='', encoding='UTF8') as f:
        writer = csv.writer(f)
        writer.writerow(data)


def redirect_to_page(page_number):

    page_url = 'https://windows10spotlight.com/page/' + str(page_number)

    page_url_data = requests.get(page_url).text

    page_soup = BeautifulSoup(page_url_data, 'html.parser')


    for link in page_soup.find_all('a', href = True):
        if(link.img != None):
            img_id = link.img['alt']
            if(img_id not in bing_df['ImageID'].values and img_id != ''):
                redirect_to_image(img_id)

#browse through every page

for page in range(1,number_of_pages+1):

    redirect_to_page(page) #go to the specific page number